In [ ]:
from google.cloud import bigquery

client = bigquery.Client()
query = """
    SELECT * FROM `numeric-advice-452700-j9.neo_bank_.modelo_prediccion`
"""
df_modelo_prediccion = client.query(query).to_dataframe()

df_modelo_prediccion

In [ ]:
print(df_modelo_prediccion.shape)
print(df_modelo_prediccion.dtypes)
df_modelo_prediccion.head()
df_modelo_prediccion.isnull().sum()  # Confirma que no hay nulos


# LogisticRegression

In [ ]:
cols_excluir = [
    'user_id', 
    'first_transaction_date', 
    'last_transaction_date', 
    'created_date',
    'dias_inactivo',
    'dias_a_ultima_txn'
]

features = [col for col in df_modelo_prediccion.columns if col not in cols_excluir + ['churned']]
X = df_modelo_prediccion[features]
y = df_modelo_prediccion['churned']

print(f"Features usadas para el modelo: {features}")


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Detectar variables categóricas y numéricas
categorical_features = X.select_dtypes(include='object').columns.tolist()
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Pipeline con regresión logística como modelo inicial
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])


In [ ]:
model_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

coefficients = model_pipeline.named_steps['classifier'].coef_[0]
feature_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()
feat_importance = pd.Series(coefficients, index=feature_names).sort_values(key=abs, ascending=False)

feat_importance[:20].plot(kind='barh')
plt.title("Principales features del modelo")
plt.show()


# Comparando modelos

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score

# Diccionario con los clasificadores que quieres probar
modelos = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

for nombre, clasificador in modelos.items():
    print(f"\n=== Modelo: {nombre} ===")
    
    # Crear pipeline completo con el preprocesador y el clasificador
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clasificador)
    ])
    
    # Entrenar
    pipeline.fit(X_train, y_train)
    
    # Predicciones
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:,1]
    
    # Métricas
    print(classification_report(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_proba))


# PayCaret

In [ ]:
# from pycaret.classification import *

# clf_setup = setup(
#     data=df_limpio,
#     target='churned',
#     train_size=0.7,
#     session_id=42
# )




In [ ]:
# best_model = compare_models()


In [ ]:
# df_modelo_prediccion['churned'].value_counts(normalize=True)


In [ ]:
# get_config('X_train').info()


In [ ]:
df_modelo_prediccion.isnull().sum()


# GradientBoosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Vuelve a crear el pipeline con el mejor modelo
best_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

# Entrena con los datos de entrenamiento
best_model.fit(X_train, y_train)

# Evalúa en el set de prueba
from sklearn.metrics import classification_report, roc_auc_score

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7]
}

grid_search = GridSearchCV(
    best_model,
    param_grid,
    cv=3,
    scoring='roc_auc',
    verbose=2,
    n_jobs=1  # Usa 1 núcleo para evitar problemas de paralelización
)


grid_search.fit(X_train, y_train)

print("Mejores parámetros encontrados:", grid_search.best_params_)
print("Mejor ROC AUC en validación cruzada:", grid_search.best_score_)


In [ ]:
# Actualizar el clasificador con los mejores parámetros
from sklearn.ensemble import GradientBoostingClassifier

mejor_clasificador = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=5,
    n_estimators=100,
    random_state=42
)

# Crear pipeline final
from sklearn.pipeline import Pipeline

modelo_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', mejor_clasificador)
])

# Entrenar con TODOS los datos (X_train + X_test)
modelo_final.fit(X, y)


In [ ]:
import joblib

joblib.dump(modelo_final, '../models/modelo_churn.pkl')
